# Trabajo Práctico Aprendizaje Automático 1

In [20]:
import pandas as pd
import numpy as np

Fijamos una semilla para tener reproducibilidad en los resultados

In [21]:
SEED = 42
rng = np.random.default_rng(SEED)

## 1. Separación de datos

Primero que nada guardamos los datos que vamos a usar en el trabajo.

In [22]:
data = pd.read_csv('..\\Data\\data.csv')

Luego lo que debemos hacer antes de empezar con el análisis exploratorio y el desarrollo de modelos es, dividir nuestros datos en desarrollo y control. Esto se hace con el objetivo de que al final del trabajo podamos reportar la performance esperada del modelo final con datos de la vida real.

Para hacer esto se me ocurrieron dos ideas (nada innovador):

1. Mezclar todos los datos y quedarnos con el porcentaje deseado.

2. Mezclar todos los datos y quedarnos con el porcentaje deseado pero estratificando. Nose si hacer esto es trampa, porque haciendo eso ya voy a conocer algo de los datos de control. 

Voy a implementar ambos y despues decidimos. IMPORTANTE: antes de seguir con los otros puntos elegir una forma de dividir y no volver a tocar esos datos.

In [23]:
#Defino el porcentaje que vamos a usar para control. A debatir.. me pareció mucho usar el 20%. 

porcent = 0.10

### Opción 1:

In [24]:
# Mezclamos los datos directamente del dataframe. 
data = data.sample(frac=1, random_state=SEED).reset_index(drop=True)

#Tomamos el porcentaje de control y desarrollo
n_control = int(len(data)*porcent)

#Dividimos..
data_control = data.iloc[:n_control].reset_index(drop=True)
data_dev = data.iloc[n_control:].reset_index(drop=True)

### Opción 2:

In [25]:
#Primero obtenemos los índices de cada clase en el data frame, ya que queremos estratificar. Tuve que agregar los .copy() porque me tiraba error.
indices_pos = data[data['target'] == 1].index.to_numpy().copy()
indices_neg = data[data['target'] == 0].index.to_numpy().copy()

# Luego mezclamos los índices de cada clase. 
rng.shuffle(indices_pos)
rng.shuffle(indices_neg)

# Calculamos la cantidad de instancias de control. ACA es la parte donde se estratifica.
n_control_pos = int(np.round(len(indices_pos) * porcent))
n_control_neg = int(np.round(len(indices_neg) * porcent))

# Partimos los indices de cada clase en control y desarrollo.
idx_control_pos = indices_pos[:n_control_pos]
idx_dev_pos     = indices_pos[n_control_pos:]

idx_control_neg = indices_neg[:n_control_neg]
idx_dev_neg     = indices_neg[n_control_neg:]

#Concatenamos los indices de control y desarrollo de cada clase para obtener los índices finales.
idx_control = np.concatenate([idx_control_pos, idx_control_neg])
idx_dev     = np.concatenate([idx_dev_pos, idx_dev_neg])

# Y mezclamos para que no nos queden los positivos por un lado y los negativos por el otro. Esto nose si es al pedo
rng.shuffle(idx_control)
rng.shuffle(idx_dev)

# Finalmente construimos los dataframes de desarrollo y control 
data_dev = data.loc[idx_dev].reset_index(drop=True)
data_control = data.loc[idx_control].reset_index(drop=True)

# Verificación de la estratificación
print(f"Total de datos: {len(data)}")
print(f"Desarrollo: {len(data_dev)} filas | Proporción Positivos: {data_dev['target'].mean():.4f}")
print(f"Control:    {len(data_control)} filas  | Proporción Positivos: {data_control['target'].mean():.4f}")

Total de datos: 500
Desarrollo: 450 filas | Proporción Positivos: 0.2822
Control:    50 filas  | Proporción Positivos: 0.2800
